# Chapter 59: Demand Forecasting and Inventory-Risk Planning at NRG

Build an auditable weekly replenishment decision from synthetic NRG data.

**Level:** Advanced  |  **Time:** 90 minutes  |  **Seed:** 20260828

Prerequisites: time-aware evaluation, forecasting metrics, uncertainty, and operational monitoring.


In [ ]:
from pathlib import Path
import sys,csv,runpy,numpy as np,matplotlib.pyplot as plt
root=Path.cwd().parents[1]
sys.path.insert(0,str(root/'src'))
from datasciencebook.inventory_planning import *
data_path=root/'data/processed/nrg_capstone_weekly_demand.csv'
if not data_path.exists(): runpy.run_path(root/'scripts/build_ch59_assets.py',run_name='ch59_assets')['generate_data']()
print('Imports ready.')


Imports ready.


In [ ]:
rows=list(csv.DictReader(data_path.open()))
series=np.array([float(r['latent_demand_cases']) for r in rows if r['product_id']=='SAMBAL_250'])
train,test=series[:-13],series[-13:]
print(f'Rows: {len(rows)}; products: {len(set(r["product_id"] for r in rows))}')
print(f'Sambal train/test: {len(train)}/{len(test)} weeks')


Rows: 312; products: 3
Sambal train/test: 91/13 weeks


In [ ]:
seasonal=np.resize(train[-13:],13)
drift=train[-1]+(train[-1]-train[0])/(len(train)-1)*np.arange(1,14)
seasonal_wape=wape(test,seasonal);drift_wape=wape(test,drift)
print(f'Seasonal-naive WAPE: {seasonal_wape:.3f}')
print(f'Drift WAPE: {drift_wape:.3f}')


Seasonal-naive WAPE: 0.068
Drift WAPE: 0.079


In [ ]:
errors=[]
for end in range(39,len(train)-2,2): errors.append(np.sum(train[end:end+2]-np.resize(train[end-13:end],2)))
forecast=seasonal[:2];plan=lead_time_target(forecast,errors,.90);position=inventory_position(120,24,8);order=order_up_to_quantity(plan['target'],position,12)
print(f"Two-week forecast: {plan['point_demand']:.0f} cases")
print(f"Safety stock: {plan['safety_stock']:.1f}; target: {plan['target']:.1f}")
print(f'Inventory position: {position:.0f}; order: {order:.0f} cases')


Two-week forecast: 206 cases
Safety stock: 16.8; target: 222.8
Inventory position: 136; order: 96 cases


In [ ]:
levels=np.arange(.50,.99,.05);targets=[lead_time_target(forecast,errors,x)['target'] for x in levels];scenarios=np.array(forecast.sum())+np.asarray(errors);costs=[newsvendor_cost(scenarios,t,1,4) for t in targets]
fig,axes=plt.subplots(1,2,figsize=(11,4));axes[0].plot(np.arange(len(series)),series,label='demand');axes[0].plot(np.arange(len(train),len(series)),seasonal,label='seasonal naive');axes[0].axvline(len(train)-.5,color='black',ls='--');axes[0].set(title='Weekly sambal demand',xlabel='Week',ylabel='Cases');axes[0].legend();axes[1].plot(levels,costs,marker='o');axes[1].set(title='Service-cost sensitivity',xlabel='Planning quantile',ylabel='Mean scenario cost');fig.tight_layout();plt.show()


## Decision note

The baseline wins this holdout and produces a 96-case order after uncertainty, inventory position, and case-pack constraints are applied. This is a synthetic demonstration, not an NRG production forecast. Stockout correction, supplier capacity, expiry, multiple locations, and approval controls remain necessary.


In [ ]:
# Practice: change the shortage cost and service level, then explain the policy trade-off.
